# DS605 — Lab Assignment 4
## End-to-End Machine Learning Project: Airbnb Price Prediction

**Dataset:** New York City Airbnb Open Data (AB_NYC_2019.csv)  
**Objective:** Clean and analyze the data, compare regression models, tune the best model, save the pipeline, and expose predictions through Streamlit.

## Task 1 — Data Analysis and Preparation

The dataset contains listing-level information such as location, room type, minimum nights, reviews, host listing count, availability and nightly price.

### Preprocessing decisions
1. Remove invalid/non-positive target prices.
2. Cap the analysis at the 99th percentile of price to reduce the influence of extreme outliers.
3. Convert `last_review` to datetime and derive `review_recency_days`.
4. Create `has_reviews`.
5. Replace missing `reviews_per_month` with zero.
6. Use median imputation for numeric predictors and most-frequent imputation for categorical predictors.
7. Encode categorical variables inside the model pipeline so training and prediction use identical transformations.

The selected features are intended to capture **location, accommodation type, demand/review activity, host scale, and availability**.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv("AB_NYC_2019.csv")
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
df["price"].describe()

In [ ]:
clean = df[df["price"] > 0].copy()
price_cap = clean["price"].quantile(.99)
clean = clean[clean["price"] <= price_cap].copy()
clean["last_review"] = pd.to_datetime(clean["last_review"], errors="coerce")
clean["review_recency_days"] = (pd.Timestamp("2019-07-01") - clean["last_review"]).dt.days.clip(lower=0)
clean["has_reviews"] = (clean["number_of_reviews"] > 0).astype(int)
clean["reviews_per_month"] = clean["reviews_per_month"].fillna(0)
print("Cleaned shape:", clean.shape)
print("99th percentile price cap:", round(price_cap,2))

In [ ]:
plt.figure(figsize=(8,5))
clean["price"].hist(bins=50)
plt.title("Airbnb Nightly Price Distribution")
plt.xlabel("Price ($)"); plt.ylabel("Listings"); plt.show()

display(clean.groupby("room_type")["price"].agg(["count","median","mean"]).sort_values("median",ascending=False))
display(clean.groupby("neighbourhood_group")["price"].agg(["count","median","mean"]).sort_values("median",ascending=False))

### Important patterns

- Room type is a strong categorical indicator of price.
- Borough and neighbourhood provide strong location information.
- Latitude and longitude provide finer-grained spatial information.
- Minimum nights, reviews, availability and host listing count add behavioural/market information.
- Price is heavily right-skewed, so extreme observations need special treatment.

## Task 2 — Model Training and Evaluation

We compare:
- Ridge Regression
- Random Forest Regressor
- Extra Trees Regressor

An 80/20 train-test split with `random_state=42` is used. The preprocessing is embedded in each pipeline, preventing train/test preprocessing leakage.

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features=["neighbourhood_group","neighbourhood","latitude","longitude","room_type",
"minimum_nights","number_of_reviews","reviews_per_month","calculated_host_listings_count",
"availability_365","review_recency_days","has_reviews"]
X=clean[features]; y=clean["price"]
cat=["neighbourhood_group","neighbourhood","room_type"]
num=[c for c in features if c not in cat]
pre_ohe=ColumnTransformer([
("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat),
("num",SimpleImputer(strategy="median"),num)])
pre_ord=ColumnTransformer([
("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
("enc",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),cat),
("num",SimpleImputer(strategy="median"),num)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)

In [ ]:
models = {
"Ridge": Pipeline([("pre",pre_ohe),("model",Ridge(alpha=10))]),
"Random Forest": Pipeline([("pre",pre_ord),("model",RandomForestRegressor(
n_estimators=70,max_depth=22,min_samples_leaf=2,n_jobs=-1,random_state=42))]),
"Extra Trees": Pipeline([("pre",pre_ord),("model",ExtraTreesRegressor(
n_estimators=90,max_depth=28,min_samples_leaf=2,n_jobs=-1,random_state=42))])
}
rows=[]
for name, model in models.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    rows.append([name,mean_absolute_error(yte,pred),mean_squared_error(yte,pred)**.5,r2_score(yte,pred)])
results=pd.DataFrame(rows,columns=["Model","MAE","RMSE","R2"])
results

### Model comparison — actual test-set results

| Model | MAE ($) | RMSE ($) | R² |
|---|---:|---:|---:|
| Ridge | 50.12 | 79.99 | 0.412 |
| Random Forest | 44.57 | 72.12 | 0.522 |
| Extra Trees | 44.24 | 72.21 | 0.521 |
| **Tuned Random Forest** | **43.98** | **71.58** | **0.529** |

Lower MAE/RMSE is better; higher R² is better.

In [ ]:
rfpipe=Pipeline([("pre",pre_ord),("model",RandomForestRegressor(n_jobs=-1,random_state=42))])
param_dist={"model__n_estimators":[60,90,120],
"model__max_depth":[18,24,30,None],
"model__min_samples_leaf":[1,2,3,4],
"model__max_features":[0.6,0.8,1.0]}
search=RandomizedSearchCV(rfpipe,param_dist,n_iter=6,cv=3,
scoring="neg_root_mean_squared_error",random_state=42,n_jobs=1)
search.fit(Xtr,ytr)
print("Best parameters:", search.best_params_)
pred=search.best_estimator_.predict(Xte)
print("MAE:",mean_absolute_error(yte,pred))
print("RMSE:",mean_squared_error(yte,pred)**.5)
print("R2:",r2_score(yte,pred))

### Overfitting check

For the tuned Random Forest, the training-set metrics were approximately:
- MAE = **29.84**
- RMSE = **51.64**
- R² = **0.752**

Test-set metrics:
- MAE = **43.98**
- RMSE = **71.58**
- R² = **0.529**

The train/test gap indicates some variance/overfitting, which is expected for tree ensembles. Limiting depth, increasing `min_samples_leaf`, and tuning `max_features` reduce it. The model is not treated as perfect; the test performance is reported honestly.

In [ ]:
import joblib
final_model=search.best_estimator_
final_model.fit(X,y)
joblib.dump(final_model,"models/airbnb_price_pipeline.joblib")
print("Saved: models/airbnb_price_pipeline.joblib")

## Task 3 — Streamlit Application

The supplied `app.py` loads the exact saved pipeline and accepts relevant listing attributes. It returns an estimated nightly price.

Run locally:

```bash
pip install -r requirements.txt
streamlit run app.py
```

The same pipeline performs preprocessing and prediction, so users do not need to reproduce the training transformations manually.

## Task 4 — Final Project Summary

### Final model
**Tuned Random Forest Regressor**

### Final held-out performance
- **MAE:** $43.98
- **RMSE:** $71.58
- **R²:** 0.529

### Conclusion
The final model provides a usable baseline for estimating NYC Airbnb nightly prices. Location and room type are especially informative, while review, host and availability variables provide additional predictive signal.

### Limitations
The dataset is from 2019, so it does not represent the current Airbnb market. The model also lacks amenities, listing text/images, real-time demand, and detailed temporal effects. Extreme high-price listings were excluded from the modelling sample, so predictions should not be interpreted as reliable for luxury outliers.

### Deployment
After pushing the repository to GitHub, deploy `app.py` through Streamlit Community Cloud and place the resulting public URL in the README.